# 北上广深租房市场数据分析

## 第九部分：使用DuckDB SQL复现核心业务分析

In [1]:
from pathlib import Path

import duckdb  # 执行 SQL 查询
import pandas as pd  # 只把 SQL 询结果显示成表格，不负责核心统计

PROJECT_ROOT = Path("..")  # 当前 Notebook 位于 notebooks 文件夹，.. 表示返回上一级，也就是项目根目录。
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "rent_cleaned.csv"
SQL_DIR = PROJECT_ROOT / "sql"

con = duckdb.connect(database=":memory:")  # 启动一个只存在于内存中的 DuckDB 数据库。关闭 Notebook 后它就会消失，因此不会产生需要提交的数据库文件，也不会修改CSV。

print("DuckDB版本：", duckdb.__version__)
print("清洗数据存在：", DATA_PATH.exists())
print("SQL目录存在：", SQL_DIR.exists())

DuckDB版本： 1.5.5
清洗数据存在： True
SQL目录存在： True


## 一、连接数据并验证SQL读取范围

In [2]:
csv_path = DATA_PATH.as_posix()

con.execute(f"""
    CREATE OR REPLACE VIEW rent_data AS
    SELECT *
    FROM read_csv_auto('{csv_path}', header=True)
""")

view_check = con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM rent_data
""").df()

view_check

,total_rows
0,11978


In [6]:
VALIDATION_SQL_PATH = SQL_DIR / "01_data_validation.sql"

validation_sql = VALIDATION_SQL_PATH.read_text(
    encoding="utf-8"
)

data_scope_result = con.execute(  # 让 DuckDB 执行 SQL,把查询结果转换成表格,在 Notebook 中展示
    validation_sql
).df()

data_scope_result
# rent_cleaned.csv
#         ↓
# 临时视图 rent_data
#         ↓
# 01_data_validation.sql 中的查询
#         ↓
# DuckDB执行
#         ↓
# Notebook显示结果

,total_rows,city_count,rental_type_count,beijing_rows,shanghai_rows,guangzhou_rows,shenzhen_rows,entire_rows,shared_rows,small_entire_rows,large_area_conflict_rows,low_price_entire_rows,any_existing_anomaly_rows
0,11978,4,2,2992,2995,2992,2999,11564,414,11.0,1.0,27.0,38.0


## 二、使用SQL复现城市整租市场分析

In [9]:
MARKET_SQL_PATH = SQL_DIR / "02_market_analysis.sql"

market_sql = MARKET_SQL_PATH.read_text(
    encoding="utf-8"
)

city_rent_result = con.execute(
    market_sql
).df()

city_rent_result

,city,entire_rows,avg_monthly_rent,median_monthly_rent,median_rent_per_sqm,relationship_rows,city_area_rent_corr,overall_area_rent_corr
0,北京,2992,8969.08,6500.0,93.02,2991,0.730,0.707
1,上海,2992,9364.25,6000.0,80.88,2991,0.703,0.707
2,深圳,2682,6661.09,4500.0,85.03,2670,0.743,0.707
3,广州,2898,3943.05,3100.0,47.13,2874,0.655,0.707


In [10]:
# 核对SQL结果与既有Pandas结论
market_check = city_rent_result.set_index("city")  # set_index("city")：便于按照城市核对结果

assert market_check[   # assert：条件不成立时立即报错，避免错误结果继续流入后续分析
    "median_monthly_rent"
].to_dict() == {       # to_dict()：把结果转换成“城市 → 数值”的对应关系
    "北京": 6500,
    "上海": 6000,
    "深圳": 4500,
    "广州": 3100
}

assert market_check[
    "median_rent_per_sqm"
].to_dict() == {
    "北京": 93.02,
    "上海": 80.88,
    "深圳": 85.03,
    "广州": 47.13
}

assert market_check[
    "city_area_rent_corr"
].to_dict() == {
    "北京": 0.730,
    "上海": 0.703,
    "深圳": 0.743,
    "广州": 0.655
}

assert city_rent_result[
    "entire_rows"
].sum() == 11564

assert city_rent_result[
    "relationship_rows"
].sum() == 11526

assert city_rent_result[
    "overall_area_rent_corr"
].eq(0.707).all()

print("城市市场SQL结果与既有Pandas结论一致。")

城市市场SQL结果与既有Pandas结论一致。


## 三、使用SQL复现指定预算下的整租房源分析

In [19]:
BUDGET_SQL_PATH = SQL_DIR / "03_budget_analysis.sql"

budget_sql = BUDGET_SQL_PATH.read_text(
    encoding="utf-8"
)

district_budget_result = con.execute(
    budget_sql
).df()

print(
    "行政区结果数量：",
    len(district_budget_result)
)

district_budget_result

行政区结果数量： 49


,city,city_analysis_rows,city_budget_rows,city_budget_share_pct,dist,district_analysis_rows,district_budget_rows,district_budget_share_pct,city_district_rank
0,上海,2991,1212,40.52,浦东,791,302,38.18,1
1,上海,2991,1212,40.52,松江,261,153,58.62,2
2,上海,2991,1212,40.52,嘉定,189,134,70.90,3
3,上海,2991,1212,40.52,闵行,286,112,39.16,4
4,上海,2991,1212,40.52,宝山,167,104,62.28,5
5,上海,2991,1212,40.52,青浦,173,85,49.13,6
6,上海,2991,1212,40.52,普陀,191,73,38.22,7
7,上海,2991,1212,40.52,奉贤,57,55,96.49,8
8,上海,2991,1212,40.52,徐汇,213,51,23.94,9
9,上海,2991,1212,40.52,杨浦,108,48,44.44,10


In [16]:
# 展示每个城市预算内房源数量前3名行政区
district_top3_result = district_budget_result.loc[
    district_budget_result[
        "city_district_rank"
    ] <= 3,
    [
        "city",
        "dist",
        "district_analysis_rows",
        "district_budget_rows",
        "district_budget_share_pct",
        "city_district_rank"
    ]
].reset_index(drop=True)

district_top3_result

,city,dist,district_analysis_rows,district_budget_rows,district_budget_share_pct,city_district_rank
0,上海,浦东,791,302,38.18,1
1,上海,松江,261,153,58.62,2
2,上海,嘉定,189,134,70.90,3
3,北京,通州,218,140,64.22,1
4,北京,大兴,221,135,61.09,2
5,北京,丰台,319,104,32.60,3
6,广州,白云,658,578,87.84,1
7,广州,番禺,550,479,87.09,2
8,广州,增城,362,343,94.75,3
9,深圳,宝安区,573,463,80.80,1


In [17]:
# 提取SQL返回的城市汇总结果
city_budget_check = district_budget_result[
    [
        "city",
        "city_analysis_rows",
        "city_budget_rows",
        "city_budget_share_pct"
    ]
].drop_duplicates().set_index("city")

# 核对各城市预算内房源数量
assert city_budget_check[
    "city_budget_rows"
].to_dict() == {
    "广州": 2329,
    "深圳": 1493,
    "上海": 1212,
    "北京": 861
}

# 核对各城市预算内房源占比
assert city_budget_check[
    "city_budget_share_pct"
].to_dict() == {
    "广州": 81.04,
    "深圳": 55.92,
    "上海": 40.52,
    "北京": 28.79
}

# 核对总体记录数量
assert district_budget_result[
    "district_analysis_rows"
].sum() == 11526

assert district_budget_result[
    "district_budget_rows"
].sum() == 5895

# 核对各城市排名第一的行政区
district_top1_check = district_budget_result[
    district_budget_result[
        "city_district_rank"
    ] == 1
].set_index("city")

assert district_top1_check[
    "dist"
].to_dict() == {
    "上海": "浦东",
    "北京": "通州",
    "广州": "白云",
    "深圳": "宝安区"
}

assert len(district_top3_result) == 12

print("预算分析SQL结果与第8天Pandas结论一致。")

预算分析SQL结果与第8天Pandas结论一致。


## 四、统一检查SQL文件和分析结果

In [18]:
# 统一检查三个正式SQL文件
sql_paths = [
    VALIDATION_SQL_PATH,
    MARKET_SQL_PATH,
    BUDGET_SQL_PATH
]

sql_file_checks = []

for sql_path in sql_paths:
    sql_text = sql_path.read_text(
        encoding="utf-8"
    )

    sql_result = con.execute(
        sql_text
    ).df()

    sql_file_checks.append({
        "SQL文件": sql_path.name,
        "文件存在": sql_path.exists(),
        "结果行数": len(sql_result)
    })

sql_file_check_result = pd.DataFrame(
    sql_file_checks
)

sql_file_check_result

,SQL文件,文件存在,结果行数
0,01_data_validation.sql,True,1
1,02_market_analysis.sql,True,4
2,03_budget_analysis.sql,True,49


## 数据分析总结

### 数据基本情况

- DuckDB版本为1.5.5，使用内存连接直接查询清洗后的CSV，没有生成数据库文件。
- 清洗数据共11,978条记录、26个字段，其中整租11,564条、合租414条。
- 城市市场指标使用全部整租记录；面积与租金关系分析及预算分析使用排除38条既有异常记录后的11,526条整租记录。
- 核心统计由三个独立SQL文件完成，Pandas只负责展示和核对SQL输出。

### 主要分析结果

- 城市整租月租金中位数依次为北京6,500元、上海6,000元、深圳4,500元和广州3,100元。
- 城市整租每平方米租金中位数依次为北京93.02元、深圳85.03元、上海80.88元和广州47.13元。
- 整租面积与月租金总体相关系数约为0.707，四个城市内部均为正相关；相关关系不能证明因果关系。
- 5,000元预算内共有5,895条整租房源，占预算分析样本的51.15%。
- 预算内房源数量依次为广州2,329条、深圳1,493条、上海1,212条和北京861条。
- 各城市预算内房源数量最多的行政区分别为上海浦东、北京通州、广州白云和深圳宝安区。
- 所有结果只描述当前数据样本，不代表实时全部租房市场供应。